# Week 16, Day 4: The multivariate Gaussian: covariance, marginals and conditionals

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w016/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Correlated population noise

Build the covariance of three neurons from standard deviations (1, 2, 1.5) and correlations 0.6, 0.3 and −0.2, sample 50,000 noise vectors with the Cholesky factor, and check the sample covariance. Then, for the first two neurons only, compute each sample's squared Mahalanobis distance and the fraction within distance 2.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(2006)
sd = np.array([1.0, 2.0, 1.5])
R = np.array([[1, 0.6, 0.3], [0.6, 1, -0.2], [0.3, -0.2, 1]])
Sigma = R * np.outer(sd, sd)

L = ...                                   # Cholesky factor
X = ...                                   # 50,000 samples as rows: (L @ z).T for z standard normal
S2 = Sigma[:2, :2]
d2 = ...                                  # squared Mahalanobis distance of X[:, :2] under S2
frac_within = np.mean(d2 < 4)
print("sample covariance:\n", np.cov(X.T).round(3)); print(f"fraction within Mahalanobis distance 2: {frac_within:.4f}")

In [ ]:
import numpy as np
def test_cov():
    "the sample covariance matches Sigma to within 0.05 in every entry"
    assert np.abs(np.cov(X.T) - Sigma).max() < 0.05

def test_mahal():
    "about 86.5% of two-dimensional samples lie within Mahalanobis distance 2 (1 - exp(-2))"
    assert abs(frac_within - (1 - np.exp(-2))) < 0.01

test_cov()
test_mahal()
print("All checks passed")

### Exercise: Correlations and the population sum

One hundred neurons each have noise variance 1, and every pair is correlated at 0.1. Compute the variance of the summed count `var_sum` from the formula, the variance for independent neurons `var_indep`, and check `var_sum` by simulating 20,000 trials (`var_sim`).

In [ ]:
N = 100
Sigma100 = 0.1 * np.ones((N, N)) + 0.9 * np.eye(N)
one = np.ones(N)
var_sum = ...
var_indep = ...
samples = rng.multivariate_normal(np.zeros(N), Sigma100, size=20000)
var_sim = ...
print(f"variance of the sum: correlated {var_sum:.1f}, independent {var_indep:.1f}, ratio {var_sum / var_indep:.1f};  simulated {var_sim:.1f}")

In [ ]:
import numpy as np
def test_formula():
    "var_sum equals 1' Sigma 1"
    assert abs(var_sum - Sigma100.sum()) < 1e-9

def test_ratio():
    "with correlation 0.1 among 100 neurons the variance of the sum is about 11 times that of independent neurons"
    assert abs(var_sum / var_indep - 10.9) < 0.05

def test_simulated():
    "a simulation of the sum agrees with the formula to within 3%"
    assert abs(var_sim - var_sum) / var_sum < 0.03

test_formula()
test_ratio()
test_simulated()
print("All checks passed")

### Exercise: Predict one neuron from five others

Six neurons have a random covariance `Sig`. Using the conditioning formula, compute the coefficients `coef` (Σ12 Σ22⁻¹) and the conditional variance `cond_var` of neuron 0 given neurons 1 to 5. Check both against least-squares regression of neuron 0 on the others in 50,000 samples (`beta_ls`, `resid_var`).

In [ ]:
A = rng.normal(size=(6, 6))
Sig = A @ A.T / 6 + 0.3 * np.eye(6)          # a random valid covariance
S11, S12, S21, S22 = Sig[:1, :1], Sig[:1, 1:], Sig[1:, :1], Sig[1:, 1:]

coef = ...                                   # shape (5,)
cond_var = ...                               # a number

Xs = rng.multivariate_normal(np.zeros(6), Sig, size=50000)
beta_ls = np.linalg.lstsq(Xs[:, 1:], Xs[:, 0], rcond=None)[0]
resid_var = np.var(Xs[:, 0] - Xs[:, 1:] @ beta_ls)
print("coefficients, formula:", coef.round(3)); print("coefficients, regression:", beta_ls.round(3))
print(f"marginal variance {Sig[0, 0]:.3f}, conditional variance {cond_var:.3f}, residual variance {resid_var:.3f}")

In [ ]:
import numpy as np
def test_coef():
    "the conditional-mean coefficients match least-squares regression on samples to within 0.05"
    assert np.abs(coef - beta_ls).max() < 0.05

def test_var():
    "the conditional variance is below the marginal variance and matches the regression residual variance to within 3%"
    assert cond_var < Sig[0, 0] and abs(cond_var - resid_var) / cond_var < 0.03

test_coef()
test_var()
print("All checks passed")